In [1]:
import pandas as pd
import numpy as np
import random

np.random.seed(42)
random.seed(42)

# ---------------------------------------------
# CONFIGURATION
# ---------------------------------------------
NUM_REQUESTS = 15000   # ~15000 requests
DONORS_PER_REQUEST = (5, 8)  # min, max donors per request

hospital_ids = [f"MUM-H-{i:02d}" for i in range(1, 16)]
blood_types = ["O+", "O-", "A+", "A-", "B+", "B-", "AB+", "AB-"]

urgency_levels = ["Critical", "Urgent", "Standard"]
urgency_weight_map = {"Critical": 3, "Urgent": 2, "Standard": 1}

# ---------------------------------------------
# BLOOD COMPATIBILITY RULES
# ---------------------------------------------
compatibility = {
    "O-": blood_types,
    "O+": ["O+", "A+", "B+", "AB+"],
    "A-": ["A-", "A+", "AB-", "AB+"],
    "A+": ["A+", "AB+"],
    "B-": ["B-", "B+", "AB-", "AB+"],
    "B+": ["B+", "AB+"],
    "AB-": ["AB-", "AB+"],
    "AB+": ["AB+"]
}

# ---------------------------------------------
# GENERATION
# ---------------------------------------------
rows = []

request_counter = 3000
donor_counter = 100000

for _ in range(NUM_REQUESTS):

    request_id = f"REQ-{request_counter}"
    request_counter += 1

    hospital_id = random.choice(hospital_ids)
    patient_blood = random.choice(blood_types)
    urgency = random.choice(urgency_levels)
    urgency_weight = urgency_weight_map[urgency] / 3  # normalize 0–1

    num_donors = random.randint(*DONORS_PER_REQUEST)

    candidate_list = []

    for _ in range(num_donors):

        donor_id = f"MUM-D-{donor_counter}"
        donor_counter += 1

        donor_blood = random.choice(blood_types)

        # -------------------------
        # Compatibility Bonus
        # -------------------------
        if donor_blood == patient_blood:
            compatibility_bonus = 1
            is_exact = 1
        elif patient_blood in compatibility.get(donor_blood, []):
            compatibility_bonus = 0.5
            is_exact = 0
        else:
            compatibility_bonus = 0
            is_exact = 0

        # -------------------------
        # Distance (km)
        # -------------------------
        distance = round(np.random.exponential(scale=8), 2)
        proximity_score = round(1 / (1 + distance), 3)

        # -------------------------
        # Availability (Beta distribution)
        # -------------------------
        if random.random() > 0.5:
            availability = np.random.beta(5, 2)
        else:
            availability = np.random.beta(2, 5)

        availability = round(float(availability), 3)

        # -------------------------
        # Final Match Score
        # -------------------------
        final_score = (
            0.35 * compatibility_bonus +
            0.25 * availability +
            0.20 * proximity_score +
            0.20 * urgency_weight
        )

        if compatibility_bonus == 0:
            final_score *= 0.2  # penalize incompatible heavily

        final_score = round(final_score, 4)

        candidate_list.append([
            request_id,
            hospital_id,
            donor_id,
            patient_blood,
            donor_blood,
            compatibility_bonus,
            is_exact,
            urgency,
            urgency_weight,
            distance,
            proximity_score,
            availability,
            final_score
        ])

    # -----------------------------------------
    # Rank donors within request
    # -----------------------------------------
    candidate_list.sort(key=lambda x: x[-1], reverse=True)

    for rank, row in enumerate(candidate_list, start=1):
        row.append(rank)
        rows.append(row)

# ---------------------------------------------
# CREATE DATAFRAME
# ---------------------------------------------
columns = [
    "Request_ID",
    "Hospital_ID",
    "Matched_Donor_ID",
    "Patient_Blood_Type",
    "Donor_Blood_Type",
    "Compatibility_Bonus",
    "Is_Exact_Match",
    "Urgency_Level",
    "Urgency_Weight",
    "Distance_KM",
    "Proximity_Score",
    "Donor_Availability_Prob",
    "Final_Match_Score",
    "Rank"
]

df = pd.DataFrame(rows, columns=columns)

df.to_csv("synthetic_blood_matching_dataset.csv", index=False)

print("Dataset Generated")
print("Total Rows:", len(df))

Dataset Generated
Total Rows: 97966
